# 03 · Genie Code

Use Genie Code in the notebook to write a **reusable** duplicate rule once. Ask for the plan first. Ask for a SQL function or job — not a one-off answer.

**Single definition of duplicate** (reuse this everywhere):

> Same `pet_id + clinic_id + visit_date`. Earliest `submitted_date` is the original. Invoice number is not the key.

**Ends with:** a scheduled job that runs `flag_duplicate_claims()` (you build it — no starter files).

If Unity AI Gateway / Genie Code CLI is not enabled, stay in the workspace. The workshop still works.


In [ ]:
%run ./00_setup


## Canonical function (reference)

After Genie Code drafts something, converge on this shape. Genie Agent example SQL and any job should call the **same** rule.


In [ ]:
# Reference implementation — attendees should rebuild via Genie Code (Lab C),
# then schedule a job that calls this (or equivalent) daily.
spark.sql(f"""
CREATE OR REPLACE FUNCTION flag_duplicate_claims()
RETURNS TABLE (
  claim_id STRING,
  original_claim_id STRING,
  pet_id STRING,
  clinic_id STRING,
  visit_date DATE,
  invoice_number STRING,
  claimed_amount DOUBLE,
  claim_status STRING,
  visit_rank INT
)
RETURN
WITH ranked AS (
  SELECT
    c.claim_id,
    FIRST_VALUE(c.claim_id) OVER (
      PARTITION BY c.pet_id, c.clinic_id, c.visit_date
      ORDER BY c.submitted_date, c.claim_id
    ) AS original_claim_id,
    c.pet_id,
    c.clinic_id,
    c.visit_date,
    c.invoice_number,
    c.claimed_amount,
    c.claim_status,
    ROW_NUMBER() OVER (
      PARTITION BY c.pet_id, c.clinic_id, c.visit_date
      ORDER BY c.submitted_date, c.claim_id
    ) AS visit_rank
  FROM {CATALOG}.{SCHEMA}.claims c
)
SELECT * FROM ranked WHERE visit_rank > 1
""")
display(spark.sql(f"SELECT * FROM flag_duplicate_claims() LIMIT 20"))
print("function ready — wire this into a daily job next")


## Workspace prompts (paste into Genie Code)

**1. Explore**

> Profile every table in `{catalog}.{schema}`: row counts, null rates, key columns, and the join keys between them.

**2. Build the rule**

> Write a reusable SQL function `flag_duplicate_claims()` that returns claims where the same pet, clinic, and visit date were already claimed. Explain why invoice number is not reliable. Match the definition: pet_id + clinic_id + visit_date, earliest submitted_date wins.

**3. Job**

> Create a Databricks Job that runs daily, calls `flag_duplicate_claims()`, and writes results to `claims_flagged`. Show me the plan first. Do not invent a different duplicate definition.

**4. Actuarial (stretch)**

> Build a monthly loss-ratio table by plan tier from `claim_payments` and `earned_premium_monthly`, plus a data-quality check.

**5. Investments (stretch)**

> Query `investment_holdings` for market-value-weighted yield and duration by asset class.


In [ ]:
print(f"Catalog.schema for prompts: {CATALOG}.{SCHEMA}")


## Your turn — Lab C · Genie Code (20 min)

Use Genie Code to build **one** of:

1. `flag_duplicate_claims()` (if you have not already) **and** a daily job that writes `claims_flagged`
2. A monthly loss-ratio table by plan tier
3. Weighted duration / yield by asset class on `investment_holdings`

Ask for the plan first. Constrain with `LIMIT` until the query is right.

**Automated outcome:** leave the duplicate job scheduled (or show the job definition ready to schedule).


## Optional CLI (skip if Gateway is not live)

```bash
curl -fsSL https://github.com/databricks/genie-code-cli/releases/latest/download/install.sh | bash
genie --version
genie
```

Same task: create the daily job against `{catalog}.{schema}` that materializes `claims_flagged` from `flag_duplicate_claims()`.
